# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the `mlcroissant` library. The dataset contains ordered logistic regression results for household adoption predictors related to indigenous and modern knowledge in rangeland management in Northern Kenya.

### Dataset Source
The dataset is provided via a Croissant schema JSON-LD file located at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
# Print available record sets
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    print('Record Set @ids:')
    for rs in metadata.record_sets:
        print(f'- {rs["@id"]}')
else:
    print('No record sets found at top-level metadata.')

## 2. Data Overview
Review available record sets and fields with their respective `@id`s. This helps you understand the structure for further access and analysis.

In [ ]:
# List available record sets and their fields with @id

record_sets = list(dataset.record_sets())

if not record_sets:
    print("No record sets were found in the metadata.")
else:
    for rs in record_sets:
        print(f"Record set: {rs['@id']}")
        if 'fields' in rs:
            print("  Fields:")
            for field in rs['fields']:
                print(f"    - {field['@id']} (type: {field.get('dataType', 'unknown')})")
        else:
            print("  No fields found.")

## 3. Data Extraction
Let's load data for each record set into a pandas DataFrame. Data extraction always references record sets and fields by their `@id`.

If the dataset defines multiple record sets, we'll extract each and show the available columns (field `@id`s).

In [ ]:
# Extract data from all record sets (by their @id) into pandas DataFrames
"""
Each record set must be referenced by its '@id'.
We list all discovered record sets and load their records into DataFrames.
"""
dataframes = {}
for record_set in dataset.record_sets():
    rs_id = record_set['@id']
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"  No records found for {rs_id}.")
if not dataframes:
    print("No dataframes created; cannot proceed with analysis.")

## 4. Exploratory Data Analysis (EDA)
We'll select a record set (by `@id`) and a numeric field (by its `@id`) for further processing. The analysis will demonstrate filtering, normalization, and grouping by another field if available.

In [ ]:
# For demonstration, choose the first record set and a numeric field (replace below if needed)

if dataframes:
    # Pick the first record set loaded
    chosen_record_set_id = list(dataframes.keys())[0]
    df = dataframes[chosen_record_set_id]
    print(f'Using record set: {chosen_record_set_id}')
    # Heuristically find a numeric column for demo
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or (df[col][pd.notnull(df[col])].apply(lambda x: isinstance(x, (int, float))).all() if not df[col].empty else False)]
    if not numeric_fields:
        print("No numeric fields found for demonstration.")
    else:
        numeric_field_id = numeric_fields[0]
        print(f"Numeric field selected: {numeric_field_id}")
        # Filter: values greater than threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold} (showing up to 3 records):")
        display(filtered_df.head(3))

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} (first 3 records):")
        display(filtered_df[[numeric_field_id, norm_col]].head(3))

        # Grouping by a non-numeric field (if any)
        non_numeric_fields = [col for col in df.columns if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col])]
        if non_numeric_fields:
            group_field_id = non_numeric_fields[0]
            print(f"Grouping by {group_field_id} and calculating mean of {numeric_field_id}:")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            display(grouped.head(3))
        else:
            print("No non-numeric field found for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of a selected numeric field and its relationship with a group field (if possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id is available, make a boxplot
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No data loaded for visualization.")

## 6. Conclusion
In this notebook, we showed how to load and explore the FAIR² dataset using only `@id` references for record sets and fields:

- Loaded dataset metadata and records using `mlcroissant`
- Identified available record sets and field `@id`s
- Loaded tables into pandas DataFrames by record set `@id`
- Performed EDA including filtering, normalization, and grouping by key field `@id`
- Visualized numeric distributions and summary group statistics

This workflow provides a reproducible template for exploring any Croissant dataset using the `mlcroissant` library and referencing all entities by their `@id`.